In [1]:
import torch
import torch.nn as nn

class Seq2SeqEncoder(nn.Module):
    """Bộ mã hóa RNN cho mô hình Sequence-to-Sequence sử dụng GRU."""
    
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super(Seq2SeqEncoder, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # Multilayer GRU
        # Input của GRU tại 1 bước thời gian sẽ là các vector đặc trưng có kích thước embed_size
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers, dropout=dropout)

    def forward(self, X):
        # X: (batch_size, num_steps)
        # batch_size: số lượng chuỗi (câu) trong một lô dữ liệu.
        # num_steps: số lượng token trong mỗi chuỗi.
        
        # EMBEDDING LAYER
        X = self.embedding(X)
        # OUTPUT SHAPE SAU EMBEDDING: (batch_size, num_steps, embed_size)
        # Mỗi token giờ đây đã biến thành một vector độ dài embed_size.

        # CHUẨN BỊ ĐẦU VÀO CHO GRU
        # Trong PyTorch, mạng RNN/GRU mặc định mong đợi input có dạng: 
        # (thời_gian, kích_thước_lô, số_chiều_đặc_trưng) 
        # Do đó, ta cần hoán đổi (permute) trục 0 (batch_size) và trục 1 (num_steps).
        X = X.permute(1, 0, 2)
        # OUTPUT SHAPE SAU PERMUTE: (num_steps, batch_size, embed_size)

        # 4. ĐI QUA MẠNG GRU
        # Khi không truyền trạng thái ẩn ban đầu h_0 vào, PyTorch sẽ mặc định h_0 là ma trận 0.
        output, state = self.rnn(X)
        
        # OUTPUT SHAPE CỦA `output`: (num_steps, batch_size, num_hiddens)
        # -> `output` chứa các trạng thái ẩn ở TẤT CẢ các bước thời gian (chỉ lấy ở lớp GRU cuối cùng).
        # (Tương ứng với các h_1, ..., h_T trong phương trình 10.7.2)
        
        # OUTPUT SHAPE CỦA `state`: (num_layers, batch_size, num_hiddens)
        # -> `state` chứa trạng thái ẩn CUỐI CÙNG ở tất cả các lớp GRU.
        # (Tương ứng với h_T nhưng ở tất cả các lớp)

        return output, state

In [2]:
# Cấu hình các siêu tham số
vocab_size = 1000   # Kích thước từ vựng
embed_size = 256    # Số chiều của vector nhúng
num_hiddens = 512   # Số lượng node ẩn trong GRU
num_layers = 2      # Số lớp GRU (Multilayer)
batch_size = 4      # Kích thước lô (chạy 4 câu cùng lúc)
num_steps = 7       # Mỗi câu dài 7 token

# Khởi tạo mô hình
encoder = Seq2SeqEncoder(vocab_size, embed_size, num_hiddens, num_layers)

# Tạo một tensor đầu vào ngẫu nhiên (chứa các index của từ vựng từ 0 đến 999)
# Shape: (batch_size, num_steps)
X = torch.zeros((batch_size, num_steps), dtype=torch.long)

# Truyền qua bộ mã hóa
output, state = encoder(X)

print(f"Input shape X              : {X.shape}")
print(f"Output shape (tất cả h_t)  : {output.shape}") 
print(f"State shape (biến ngữ cảnh): {state.shape}")

Input shape X              : torch.Size([4, 7])
Output shape (tất cả h_t)  : torch.Size([7, 4, 512])
State shape (biến ngữ cảnh): torch.Size([2, 4, 512])


In [3]:
import torch
import torch.nn as nn

class Seq2SeqDecoder(nn.Module):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super(Seq2SeqDecoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # Input của GRU bây giờ là (embed_size + num_hiddens) 
        # vì ta luôn muốn truyền cả vector nhúng của từ và vector ngữ cảnh c vào GRU tại mỗi bước thời gian.
        self.rnn = nn.GRU(embed_size + num_hiddens, num_hiddens, num_layers, dropout=dropout)
        
        # Lớp đầu ra để dự đoán từ tiếp theo
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs):
        # Lấy trạng thái ẩn cuối cùng của Encoder làm trạng thái ẩn ban đầu cho Decoder
        # enc_outputs[1] chính là 'state' từ Encoder (h_T)
        return enc_outputs[1]

    def forward(self, X, state):
        # 1. INPUT SHAPE X: (batch_size, num_steps)
        # 2. CHUYỂN QUA EMBEDDING: (batch_size, num_steps, embed_size)
        X = self.embedding(X).permute(1, 0, 2)
        # Shape X sau permute: (num_steps, batch_size, embed_size)

        # 3. TRÍCH XUẤT BIẾN NGỮ CẢNH c
        # Biến ngữ cảnh c thường là trạng thái ẩn lớp cuối cùng của encoder
        # state[-1] shape: (batch_size, num_hiddens)
        context = state[-1]
        
        # 4. "SAO CHÉP" c CHO MỌI BƯỚC THỜI GIAN
        # Để nối được với X, context cần có cùng num_steps
        # Shape context sau repeat: (num_steps, batch_size, num_hiddens)
        context = context.repeat(X.shape[0], 1, 1)

        # 5. CONCATENATE (NỐI) X VÀ CONTEXT
        # Kết quả shape: (num_steps, batch_size, embed_size + num_hiddens)
        X_and_context = torch.cat((X, context), 2)

        # 6. ĐƯA VÀO RNN VỚI TRẠNG THÁI ẨN CŨ (state)
        output, state = self.rnn(X_and_context, state)
        
        # 7. DỰ ĐOÁN TỪ (Qua lớp Dense/Fully Connected)
        # output shape: (num_steps, batch_size, num_hiddens)
        # Chuyển về: (num_steps, batch_size, vocab_size)
        output = self.dense(output).permute(1, 0, 2)
        
        # Trả về output để tính loss và state để dùng cho bước thời gian tiếp theo
        return output, state

In [4]:
vocab_size_encoder = 1000  # Kích thước từ điển ngôn ngữ nguồn
vocab_size_decoder = 1500  # Kích thước từ điển ngôn ngữ đích

# Kích thước vector nhúng có thể khác nhau, nhưng thường đặt giống nhau cho tiện
embed_size = 256       

# BẮT BUỘC num_hiddens và num_layers phải GIỐNG NHAU giữa Encoder và Decoder
num_hiddens = 512      
num_layers = 2         

batch_size = 4         # Chạy 4 câu cùng lúc
num_steps_encoder = 7  # Độ dài câu đầu vào (ví dụ: 7 từ)
num_steps_decoder = 9  # Độ dài câu đầu ra (ví dụ: 9 từ)

# 2. Khởi tạo mô hình
encoder = Seq2SeqEncoder(vocab_size_encoder, embed_size, num_hiddens, num_layers)
decoder = Seq2SeqDecoder(vocab_size_decoder, embed_size, num_hiddens, num_layers)

# 3. Tạo dữ liệu đầu vào giả lập (chứa các index từ vựng ngẫu nhiên)
# Shape: (batch_size, num_steps)
X_enc = torch.zeros((batch_size, num_steps_encoder), dtype=torch.long)
X_dec = torch.zeros((batch_size, num_steps_decoder), dtype=torch.long)

# 4. Chạy luồng dữ liệu (Forward pass)
# BƯỚC A: Chạy Encoder
enc_outputs = encoder(X_enc)
enc_output, enc_state = enc_outputs

# BƯỚC B: Khởi tạo trạng thái cho Decoder từ kết quả của Encoder
dec_state = decoder.init_state(enc_outputs)

# BƯỚC C: Chạy Decoder
dec_output, dec_state_final = decoder(X_dec, dec_state)

# 5. In kết quả để xem Shape
print("--- SHAPE CỦA ENCODER ---")
print(f"X_enc (Input Encoder)       : {X_enc.shape}")
print(f"enc_output (Tất cả h_t)     : {enc_output.shape} -> (num_steps_enc, batch, num_hiddens)")
print(f"enc_state (Biến ngữ cảnh c) : {enc_state.shape} -> (num_layers, batch, num_hiddens)")

print("\n--- TRUYỀN TRẠNG THÁI ---")
print(f"dec_state ban đầu           : {dec_state.shape} -> Khớp hoàn toàn với enc_state!")

print("\n--- SHAPE CỦA DECODER ---")
print(f"X_dec (Input Decoder)       : {X_dec.shape}")
print(f"dec_output (Kết quả dự đoán): {dec_output.shape} -> (batch, num_steps_dec, vocab_size_decoder)")
print(f"dec_state_final (State cuối): {dec_state_final.shape} -> Dùng cho bước tiếp theo nếu cần")

--- SHAPE CỦA ENCODER ---
X_enc (Input Encoder)       : torch.Size([4, 7])
enc_output (Tất cả h_t)     : torch.Size([7, 4, 512]) -> (num_steps_enc, batch, num_hiddens)
enc_state (Biến ngữ cảnh c) : torch.Size([2, 4, 512]) -> (num_layers, batch, num_hiddens)

--- TRUYỀN TRẠNG THÁI ---
dec_state ban đầu           : torch.Size([2, 4, 512]) -> Khớp hoàn toàn với enc_state!

--- SHAPE CỦA DECODER ---
X_dec (Input Decoder)       : torch.Size([4, 9])
dec_output (Kết quả dự đoán): torch.Size([4, 9, 1500]) -> (batch, num_steps_dec, vocab_size_decoder)
dec_state_final (State cuối): torch.Size([2, 4, 512]) -> Dùng cho bước tiếp theo nếu cần


In [5]:
class Seq2Seq(nn.Module):
    """Lớp Model kết hợp Encoder và Decoder."""
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X):
        # 1. Chạy đầu vào ngôn ngữ nguồn qua Encoder
        enc_outputs = self.encoder(enc_X)
        
        # 2. Khởi tạo trạng thái ban đầu cho Decoder
        dec_state = self.decoder.init_state(enc_outputs)
        
        # 3. Chạy đầu vào ngôn ngữ đích qua Decoder
        # (Trong lúc huấn luyện, dec_X chính là câu đích nhưng bị dịch đi 1 token - Teacher Forcing)
        return self.decoder(dec_X, dec_state)

In [12]:
!pip install torch==2.2.0 torchtext==0.17.2 torchvision==0.17.1 torchaudio==2.2.1 -q

ERROR: Cannot install torch==2.2.0 and torchtext==0.17.2 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchtext.datasets import Multi30k
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

# ==========================================
# 1. CHUẨN BỊ TOKENIZER VÀ TỪ VỰNG (VOCAB)
# ==========================================
# Sử dụng tokenizer cơ bản (bạn có thể dùng spacy nếu đã cài đặt)
tokenizer_de = get_tokenizer('basic_english') # Tạm dùng basic_english cho tiếng Đức để demo chạy nhanh
tokenizer_en = get_tokenizer('basic_english')

# Các token đặc biệt
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

def yield_tokens(data_iter, language):
    for data_sample in data_iter:
        if language == 'de':
            yield tokenizer_de(data_sample[0])
        else:
            yield tokenizer_en(data_sample[1])

# Tải dữ liệu train và xây dựng từ vựng
train_iter = Multi30k(split='train', language_pair=('de', 'en'))
vocab_de = build_vocab_from_iterator(yield_tokens(train_iter, 'de'), min_freq=2, specials=special_symbols)
vocab_en = build_vocab_from_iterator(yield_tokens(train_iter, 'en'), min_freq=2, specials=special_symbols)

vocab_de.set_default_index(UNK_IDX)
vocab_en.set_default_index(UNK_IDX)

# ==========================================
# 2. XỬ LÝ DỮ LIỆU ĐẦU VÀO (COLLATE_FN)
# ==========================================
def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        # Chuyển chữ thành số (index) và thêm token <bos>, <eos>
        src_tokens = [BOS_IDX] + vocab_de(tokenizer_de(src_sample)) + [EOS_IDX]
        tgt_tokens = [BOS_IDX] + vocab_en(tokenizer_en(tgt_sample)) + [EOS_IDX]
        
        src_batch.append(torch.tensor(src_tokens, dtype=torch.long))
        tgt_batch.append(torch.tensor(tgt_tokens, dtype=torch.long))
    
    # Pad các câu trong batch cho bằng nhau (batch_first=True trả về shape: batch_size, num_steps)
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_batch, tgt_batch

# Khởi tạo DataLoader
train_iter = Multi30k(split='train', language_pair=('de', 'en'))
dataloader = DataLoader(train_iter, batch_size=32, collate_fn=collate_fn)

# ==========================================
# 3. KHỞI TẠO MODEL VÀ LOSS FUNCTION
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# (Giả sử bạn đã chạy các class Seq2SeqEncoder, Seq2SeqDecoder, Seq2Seq ở trên)
embed_size = 256
num_hiddens = 512
num_layers = 2
dropout = 0.5

encoder = Seq2SeqEncoder(len(vocab_de), embed_size, num_hiddens, num_layers, dropout)
decoder = Seq2SeqDecoder(len(vocab_en), embed_size, num_hiddens, num_layers, dropout)
model = Seq2Seq(encoder, decoder).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

# Hàm Loss phớt lờ các token <pad>
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# ==========================================
# 4. VÒNG LẶP HUẤN LUYỆN (TRAINING LOOP)
# ==========================================
model.train()
epoch_loss = 0

print("Bắt đầu huấn luyện...")
for i, (src, tgt) in enumerate(dataloader):
    src, tgt = src.to(device), tgt.to(device)
    
    # Chuẩn bị input và target cho Decoder (Kỹ thuật Teacher Forcing)
    # dec_input: Bắt đầu từ <bos>, bỏ token <eos> cuối cùng
    # dec_target: Bỏ token <bos> đầu tiên, dùng để tính Loss
    dec_input = tgt[:, :-1] 
    dec_target = tgt[:, 1:] 
    
    optimizer.zero_grad()
    
    # Truyền qua mạng (Mô hình trả về shape: batch_size, num_steps, vocab_size)
    output, _ = model(src, dec_input)
    
    # Đổi shape của output thành (batch_size, vocab_size, num_steps) để phù hợp với hàm Loss
    output = output.permute(0, 2, 1)
    
    # Tính toán lỗi
    loss = criterion(output, dec_target)
    
    # Lan truyền ngược và cập nhật trọng số
    loss.backward()
    
    # Gradient clipping (tránh hiện tượng exploding gradient trong RNN)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()
    
    epoch_loss += loss.item()
    
    if i % 100 == 0:
        print(f"Batch: {i} | Loss hiện tại: {loss.item():.4f}")

print(f"Hoàn thành 1 Epoch! Loss trung bình: {epoch_loss / len(dataloader):.4f}")

OSError: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchtext/lib/libtorchtext.so